# 第 7 周练习：从微调模型获得更好的价格预测

## 练习目标

我们已经用 **QLoRA** 在约 20K 条产品列表上微调了 **Llama 3.2-3B**，用来预测价格。  
Hub 模型：`mcaleb/price-2026-03-05_04.52.30-lite`

本练习**不再重新训练**，而是对比三种推理技巧，并把结果与前沿 API 模型对照：

1. **贪婪解码（Greedy）**：每次选概率最高的下一个 token
2. **集成采样（Ensemble）**：低温采样多次，取中位数
3. **Top-K 概率混合**：看第一个价格 token 的分布，做加权平均

## 和本课 Week 7 的关系

| 概念 | 本练习里你会看到 |
|------|------------------|
| 加载 QLoRA 适配器 | `PeftModel.from_pretrained(base, HUB_MODEL_NAME)` |
| 4bit 量化推理 | `BitsAndBytesConfig(load_in_4bit=True, ...)` |
| 解码策略对比 | greedy / sample+median / top-k blend |
| 与前沿模型比 MAE | OpenAI + OpenRouter 上的 GPT / Claude / Gemini |

## 怎么跑

1. 建议在 **Google Colab（GPU）** 运行；先跑依赖安装单元格
2. 在 Colab Secrets 配好 `HF_TOKEN`、`OPENAI_API_KEY`、`OPENROUTER_API_KEY`
3. 自上而下执行；评估会跑约 200 条测试样本并画图


In [ ]:
# ========== 安装依赖并下载课程 util.py ==========
# pip 静默升级：bitsandbytes（量化）、trl、openai（前沿模型客户端）
!pip install -q --upgrade bitsandbytes trl openai
# 从课程仓库拉取 week7/util.py（含 Tester 评估工具），保存为当前目录 util.py
!wget -q https://raw.githubusercontent.com/ed-donner/llm_engineering/main/week7/util.py -O util.py


In [ ]:
# ========== 导入 + 配置常量 + 评估封装 ==========
# 正则：从生成文本里抽数字价格
import re
# 统计：算中位数（ensemble）
import statistics
# Colab Secrets：读 HF / API 密钥，避免写进笔记本
from google.colab import userdata
# Hugging Face Hub 登录
from huggingface_hub import login
# PyTorch 张量与 CUDA
import torch
# F.softmax：把 logits 变成概率
import torch.nn.functional as F
# 因果 LM、分词器、4bit 配置、固定随机种子
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, set_seed
# 加载 Hugging Face 数据集
from datasets import load_dataset
# 把 LoRA/QLoRA 适配器挂到基座模型上
from peft import PeftModel
# 课程提供的评估器：画散点图并收集绝对误差
from util import Tester

# --- 配置：基座、作者 HF 用户名、数据集、本次 run 名 ---
# 未微调的基座模型 id（必须与训练时一致）
BASE_MODEL = "meta-llama/Llama-3.2-3B"
# 推模型到 Hub 时用的用户名
HF_USER = "mcaleb"
# 带 prompt 字段的定价数据集（lite 版）
DATASET_NAME = "ed-donner/items_prompts_lite"
# 本次微调 run 的时间戳后缀
RUN_NAME = "2026-03-05_04.52.30-lite"
# Hub 上完整微调模型名：{user}/price-{run}
HUB_MODEL_NAME = f"{HF_USER}/price-{RUN_NAME}"

# 评估样本数：越大越稳，但更慢
SIZE = 200

def run_eval(predictor, data, title=None, size=SIZE):
    """Run the course's Tester (shows charts) and return the average error."""
    # 构造 Tester：传入预测函数、数据、标题、样本量
    t = Tester(predictor, data, title=title, size=size)
    # 跑完全部样本并出图
    t.run()
    # 返回平均绝对误差 MAE
    return sum(t.errors) / t.size


In [ ]:
# ========== 登录 Hugging Face 并加载测试集 ==========
# 从 Colab Secrets 读取 HF_TOKEN
hf_token = userdata.get("HF_TOKEN")
# 登录 Hub（便于拉私有/门禁模型）；写入 git credential
login(hf_token, add_to_git_credential=True)

# 按 DATASET_NAME 拉取整份 dataset dict
dataset = load_dataset(DATASET_NAME)
# 只要 test split，后面所有评估都用它
test = dataset["test"]


## 加载微调后的模型

用 **4bit 量化**加载 Llama 3.2-3B 基座，再叠上我们训练好的 **QLoRA 适配器**。  
这样显存占用小，又能保留微调后的定价行为。


In [ ]:
# ========== 4bit 量化基座 + 挂载 QLoRA 适配器 ==========
# 读 GPU 算力主版本号：Ampere(8+) 更适合 bfloat16
capability = torch.cuda.get_device_capability()
# SM >= 8 用 bf16 计算，否则退回 fp16
use_bf16 = capability[0] >= 8

# BitsAndBytes 4bit 配置：NF4 + 双重量化，省显存
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16 if use_bf16 else torch.float16,
    bnb_4bit_quant_type="nf4",
)

# 从 Hub 加载与基座匹配的分词器
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
# pad 用 eos，避免缺 pad_token 报错
tokenizer.pad_token = tokenizer.eos_token
# 右填充：因果 LM 训练/批推理常见设置
tokenizer.padding_side = "right"

# 4bit 加载基座；device_map=auto 自动切到可用 GPU
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map="auto",
)
# 生成配置里补上 pad_token_id，减少告警
base_model.generation_config.pad_token_id = tokenizer.pad_token_id

# 从 Hub 拉取 QLoRA 适配器并挂到基座上
fine_tuned_model = PeftModel.from_pretrained(base_model, HUB_MODEL_NAME)
# 打印大约占用的显存（MB）
print(f"Memory footprint: {fine_tuned_model.get_memory_footprint() / 1e6:.1f} MB")


## 方法 1：贪婪解码——每次只选最优 token

最简单的预测方式：模型读完产品说明后，**逐步选择概率最高的下一个 token**。  
最先生成出来的数字串就是价格答案。实现快，但容易被单次「倒霉」路径拖累。


In [ ]:
# ========== 贪婪预测函数 + 跑评估 ==========
def greedy_predict(item):
    """Feed the prompt to the model, let it generate a price."""
    # 把样本的 prompt 编成张量并放到 GPU
    inputs = tokenizer(item["prompt"], return_tensors="pt").to("cuda")
    # 推理模式：不建计算图，省显存
    with torch.no_grad():
        # 默认贪婪生成，最多再写 8 个新 token（价格很短）
        output_ids = fine_tuned_model.generate(**inputs, max_new_tokens=8)
    # 提示部分长度：后面 decode 时只取「新生成」段
    prompt_len = inputs["input_ids"].shape[1]
    # 解码新生成的 token 成字符串（含价格）
    return tokenizer.decode(output_ids[0, prompt_len:])

# 收集各方法的 MAE
results = {}
# 固定种子，方便复现对比
set_seed(42)
# 跑贪婪基线，标题写入图表
results["QLoRA Greedy"] = run_eval(greedy_predict, test, "QLoRA Greedy", SIZE)


## 方法 2：多次询问，取中间答案（Ensemble）

不只问模型一次，而是带一点随机性（`temperature=0.2`）问 **3 次**，再取**中位数**。  
这样可以平滑掉偶尔特别离谱的单次猜测，通常比纯贪婪更稳。


In [ ]:
# ========== 低温多次采样 + 中位数集成 ==========
def ensemble_predict(item, n_samples=3, temperature=0.2):
    """Generate several predictions with some randomness, return the median."""
    # 编码提示并放到 GPU
    inputs = tokenizer(item["prompt"], return_tensors="pt").to("cuda")
    # 收集每次解析出的价格
    prices = []
    # 采样 n_samples 次
    for _ in range(n_samples):
        with torch.no_grad():
            # do_sample=True + temperature：允许轻微随机，而不是纯贪婪
            output_ids = fine_tuned_model.generate(
                **inputs, max_new_tokens=8,
                do_sample=True, temperature=temperature,
            )
        # 只解码新生成段
        text = tokenizer.decode(output_ids[0, inputs["input_ids"].shape[1]:])
        # 从生成的文本中提取数字
        match = re.search(r"\d+\.?\d*", text)
        if match:
            # 转成 float 加入列表
            prices.append(float(match.group()))
    # 有有效价格则返回中位数字符串；否则 "0"
    return str(statistics.median(prices)) if prices else "0"

# 固定种子后再评估，保证可复现
set_seed(42)
results["QLoRA Ensemble"] = run_eval(ensemble_predict, test, "QLoRA Ensemble", SIZE)


## 方法 3：混合模型对「第一个价格 token」的最高猜测（Top-K）

不完整生成一段文本，而是直接看模型在**下一个 token** 上的概率分布：  
取 Top-K 个候选，筛出能解析成有效数字的，再按置信度做**加权平均**。  
直觉上像在问：「你最好的几个猜测是什么？各自有多确定？」


In [ ]:
# 需要 float32 进行 logit 数学（就地更改模型）
# 把权重转到 float32，避免 4bit/半精度下做 softmax 数值不稳
fine_tuned_model = fine_tuned_model.float()

def topk_predict(item, k=3):
    """Look at the model's top-K token probabilities and blend them."""
    # 编码提示
    inputs = tokenizer(item["prompt"], return_tensors="pt").to("cuda")
    with torch.no_grad():
        # 前向一次，拿到每个位置的 logits
        outputs = fine_tuned_model(**inputs)
        # 只要「最后一个输入位置」预测下一 token 的那一行，并搬到 CPU
        logits = outputs.logits[:, -1, :].to("cpu")

    # logits → 概率分布
    probs = F.softmax(logits, dim=-1)
    # 取概率最高的 k 个 token 及其概率
    top_probs, top_ids = probs.topk(k)

    # 仅保留解码为有效价格的代币
    prices, weights = [], []
    for i in range(k):
        # 第 i 名候选 token 解码成字符串并去掉空白
        token = tokenizer.decode(top_ids[0][i]).strip()
        try:
            # 尝试解析成数字价格
            price = float(token)
            if price > 0:
                prices.append(price)
                # 对应概率作为权重
                weights.append(top_probs[0][i])
        except ValueError:
            # 非数字 token（如标点）跳过
            continue

    # 没有任何有效数字 → 返回 "0"
    if not prices:
        return "0"
    # 权重归一化后做加权平均，再转成字符串
    total = sum(weights)
    return str(sum(p * w / total for p, w in zip(prices, weights)).item())

set_seed(42)
results["QLoRA Top-K"] = run_eval(topk_predict, test, "QLoRA Top-K", SIZE)


## 前沿模型如何比较？

在**同一批 200 条**测试样本上，对比 4 个前沿模型：

- **GPT-4.1-nano**：OpenAI 小模型，零样本（不做本任务训练）
- **GPT-4.1-nano 微调**：同一底座，第 6 周在约 2K 样本上微调过
- **Claude Haiku**：Anthropic 快速模型，零样本（经 OpenRouter）
- **Gemini Flash**：Google 快速模型，零样本（经 OpenRouter）


In [ ]:
# ========== 前沿模型客户端 + 统一预测工厂 ==========
# OpenAI 官方 SDK（也用来打 OpenRouter 的兼容接口）
from openai import OpenAI

# 直连 OpenAI：用 Colab Secret 里的 OPENAI_API_KEY
openai_client = OpenAI(api_key=userdata.get("OPENAI_API_KEY"))
# OpenRouter：同一 SDK，只改 base_url 与密钥
openrouter_client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=userdata.get("OPENROUTER_API_KEY"),
)

def extract_summary(item):
    # 从课程 prompt 模板里夹出产品摘要正文
    p = item["prompt"]
    return p.split("What does this cost to the nearest dollar?\n\n")[1].split("\n\nPrice is $")[0]

def make_frontier_predictor(client, model):
    # 闭包：固定 client + model，返回可交给 Tester 的 predict(item)
    def predict(item):
        summary = extract_summary(item)
        # 聊天补全：只要美元数字，温度 0 求稳
        r = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content":
                f"Estimate the price of this product. Respond with just the dollar amount, no explanation.\n\n{summary}"}],
            max_tokens=10,
            temperature=0,
        )
        # 取助手回复文本
        return r.choices[0].message.content
    # 给函数起个短名字，方便 Tester 当标题用
    predict.__name__ = model.split("/")[-1].split(":")[0]
    return predict

# 名称 → (客户端, model id)；model id 字符串必须保持原样
frontier_models = {
    "GPT-4.1-nano": (openai_client, "gpt-4.1-nano"),
    "GPT-4.1-nano FT": (openai_client, "ft:gpt-4.1-nano-2025-04-14:personal:pricer-2000:DFdADQPs"),
    "Claude Haiku": (openrouter_client, "anthropic/claude-3.5-haiku"),
    "Gemini Flash": (openrouter_client, "google/gemini-2.5-flash-lite"),
}


In [ ]:
# ========== 逐个前沿模型评估并打印 MAE ==========
for name, (client, model) in frontier_models.items():
    # 为当前 model 造预测函数
    predictor = make_frontier_predictor(client, model)
    # 每次评估前固定种子
    set_seed(42)
    # 写入 results，标题用可读名称
    results[name] = run_eval(predictor, test, name, SIZE)
    # 控制台再打一行 MAE，方便扫结果
    print(f"{name}: ${results[name]:.2f}")


## 最终比较

把 QLoRA 三种解码策略与前沿模型的 **MAE（平均绝对误差，美元）** 画成柱状图，并打印排序表。


In [ ]:
# ========== 柱状图对比 + 排序打印 MAE ==========
# Plotly：交互式柱状图
import plotly.graph_objects as go

# x 轴标签、y 轴 MAE
labels = list(results.keys())
values = list(results.values())
# 配色：QLoRA 红、微调 FT 浅蓝、其余板岩蓝
colors = ["red" if "QLoRA" in k else "skyblue" if "FT" in k else "slateblue" for k in labels]

# 画柱状图
fig = go.Figure(go.Bar(x=labels, y=values, marker_color=colors))
fig.update_layout(
    title="Price Prediction: Mean Absolute Error by Model",
    yaxis=dict(title="MAE ($)", range=[0, max(values) * 1.1]),
    xaxis=dict(tickangle=-30),
    width=900,
    height=500,
)
# 在笔记本中展示
fig.show()

# 表头：按 MAE 从低到高排序打印
print(f"\n{'Model':<25} {'MAE':>10}")
print("-" * 37)
for name, mae in sorted(results.items(), key=lambda x: x[1]):
    print(f"{name:<25} ${mae:>8.2f}")


## 我们学到了什么

- **贪婪解码**（约 $65）是可靠基线，但仍有提升空间。
- **Ensemble**（约 $57）：低温问 3 次取中位数，往往改进最大——想法简单、回报明显。
- **Top-K 混合**（约 $57）：直接读概率分布，机制不同，效果可与 Ensemble 接近。
- 这两种技巧能把小型 QLoRA 模型推到与 **Claude Haiku / Gemini Flash** 相近的误差带，且无按次 API 成本。
- **微调仍然很强**：GPT-4.1-nano FT（约 $54）依旧领先，说明哪怕很小的前沿模型，任务微调后也很难被轻易超越。

> 具体数字以你本机/Colab 跑出的 `results` 为准；上文为作者实验量级参考。
